# Best SP vs. Pileup-Corrected $T(\mu)$

Single-region Quadrant Analysis and Feature Distribution comparison.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import mplhep as mh
mh.style.use("ATLAS")

from src.core.Plotting.Localization import PlotTranslator
from src.core.Validation.SingleRegionComparisonRunner import (
    ComparisonModelConfig,
    SingleRegionComparisonRunner,
)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
TARGET_ET: int = 1
TARGET_ETA: int = 1
REGION_KEY: str = f"et_{TARGET_ET}_eta_{TARGET_ETA}"
LANGUAGE: str = PlotTranslator.validate_language("pt")

COMPARE_SAME_MODEL: bool = True      # True: Model 1 vs Model 1 | False: Model 1 vs Model 2
TARGET_PD_MODE: str = "athena"       # 'athena' or 'custom'
CUSTOM_TARGET_PD: float = 0.9424     # Used when TARGET_PD_MODE == 'custom'
WORKING_POINT: str = "loose"         # Athena WP: 'loose', 'medium', 'tight'

MODEL_1 = ComparisonModelConfig(
    config_path=PROJECT_ROOT / "config/NeuralRinger/ModelV10_HighBatch_100Rings.yaml",
    results_path=PROJECT_ROOT / "results/config_nameModelV10_HighBatch_100Rings_id20261112210846",
    alias="Model V10 (100 Rings)",
)
MODEL_2 = ComparisonModelConfig(
    config_path=PROJECT_ROOT / "config/NeuralRinger/ModelV10_HighBatch_50Rings.yaml",
    results_path=PROJECT_ROOT / "results/config_nameModelV10_HighBatch_50Rings_id20261115051346",
    alias="Model V10 (50 Rings)",
)

DATA_PATH: Path = PROJECT_ROOT / "data/consolidated"
MANIFEST_PATH: Path = MODEL_1.results_path.parent / "split_manifest.json"
PLOT_FORMAT: str = "png"
LOG_Y: bool = True

OUTPUT_DIR: Path = PROJECT_ROOT / "Plots" / f"Comparison_best_sp_vs_pileup_{'same' if COMPARE_SAME_MODEL else 'cross'}_{REGION_KEY}"
TABLES_DIR: Path = OUTPUT_DIR / "Tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_TO_PLOT: list[str] = [
    "trig_L2_calo_et", "trig_L2_calo_eta", "trig_L2_calo_phi",
    "trig_L2_calo_e237", "trig_L2_calo_e277", "trig_L2_calo_fracs1",
    "trig_L2_calo_weta2", "trig_L2_calo_ehad1", "trig_L2_calo_emaxs1",
    "trig_L2_calo_e2tsts1", "trig_L2_calo_wstot", "Eratio",
]

In [ ]:
runner = SingleRegionComparisonRunner(
    iet=TARGET_ET,
    ieta=TARGET_ETA,
    model1=MODEL_1,
    model2=MODEL_2,
    compare_same_model=COMPARE_SAME_MODEL,
    target_pd_mode=TARGET_PD_MODE,
    custom_target_pd=CUSTOM_TARGET_PD,
    working_point=WORKING_POINT,
    data_path=DATA_PATH,
    manifest_path=MANIFEST_PATH,
    language=LANGUAGE,
    plot_format=PLOT_FORMAT,
    log_y=LOG_Y,
)
payload = runner.prepare(mode="best_sp_vs_pileup", features_to_plot=FEATURES_TO_PLOT)
print(f"Comparing: {payload.strategy1_name}  vs  {payload.strategy2_name} ({len(payload.test_labels):,} test events)")

In [ ]:
strategy_df, counts_df, saved_quad = runner.run_quadrant_analysis(payload, OUTPUT_DIR, TABLES_DIR)
display(strategy_df)
display(counts_df)
print(f"Saved {sum(len(v) for v in saved_quad.values())} quadrant plots under: {OUTPUT_DIR / 'Quadrant'}")

In [ ]:
sig_js_df, fig_sig = runner.run_variable_analysis(payload, OUTPUT_DIR, TABLES_DIR, is_signal=True)
display(sig_js_df)

In [ ]:
bg_js_df, fig_bg = runner.run_variable_analysis(payload, OUTPUT_DIR, TABLES_DIR, is_signal=False)
display(bg_js_df)